In [1]:
import kagglehub

path = kagglehub.dataset_download("jp797498e/twitter-entity-sentiment-analysis")

print("Path to dataset files:",path)

100%|██████████| 1.99M/1.99M [00:00<00:00, 3.41MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/jp797498e/twitter-entity-sentiment-analysis/versions/2


In [2]:
import os

print(os.listdir(path))

['twitter_validation.csv', 'twitter_training.csv']


In [3]:
import pandas as pd

df = pd.read_csv(os.path.join(path, "twitter_training.csv"))

df.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [4]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column                                                 Non-Null Count  Dtype 
---  ------                                                 --------------  ----- 
 0   2401                                                   74681 non-null  int64 
 1   Borderlands                                            74681 non-null  object
 2   Positive                                               74681 non-null  object
 3   im getting on borderlands and i will murder you all ,  73995 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [5]:
df.columns = ["Tweet_ID", "Entity", "Sentiment", "Tweet"]

In [6]:
df.head()

,Tweet_ID,Entity,Sentiment,Tweet
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [7]:
df['Sentiment'].value_counts()

,count
Sentiment,
Negative,22542
Positive,20831
Neutral,18318
Irrelevant,12990


In [8]:
# Preprocessing starts
df.isnull().sum()

,0
Tweet_ID,0
Entity,0
Sentiment,0
Tweet,686


In [10]:
df.dropna(inplace=True)
df['Tweet']=df['Tweet'].str.lower()

In [11]:
import re   #regular expressions

def clean_text(text):
    text = re.sub(r'http\S+', '', str(text))      # Remove URLs
    text = re.sub(r'@\w+', '', text)              # Remove mentions
    text = re.sub(r'[^a-zA-Z\s]', '', text)       # Remove special chars
    return text

df["Tweet"] = df["Tweet"].apply(clean_text)

In [12]:
# remove stopwords
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

df["Tweet"] = df["Tweet"].apply(remove_stopwords)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [13]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

def stem_text(text):
    words = text.split()
    words = [stemmer.stem(word) for word in words]
    return " ".join(words)

df["Tweet"] = df["Tweet"].apply(stem_text)

In [14]:
df[["Sentiment", "Tweet"]].head()

,Sentiment,Tweet
0,Positive,come border kill
1,Positive,im get borderland kill
2,Positive,im come borderland murder
3,Positive,im get borderland murder
4,Positive,im get borderland murder


In [15]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df["Sentiment"] = label_encoder.fit_transform(df["Sentiment"])

df["Sentiment"].head()

,Sentiment
0,3
1,3
2,3
3,3
4,3


In [16]:
# vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df["Tweet"])

y = df["Sentiment"]

print(X.shape)

(73995, 5000)


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(59196, 5000)
(14799, 5000)


In [18]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy :", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr, average='weighted'))
print("Recall   :", recall_score(y_test, y_pred_lr, average='weighted'))
print("F1 Score :", f1_score(y_test, y_pred_lr, average='weighted'))

Accuracy : 0.6801135211838638
Precision: 0.6800512633632729
Recall   : 0.6801135211838638
F1 Score : 0.6774231035290267


In [20]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()

nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)

In [21]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy :", accuracy_score(y_test, y_pred_nb))
print("Precision:", precision_score(y_test, y_pred_nb, average='weighted'))
print("Recall   :", recall_score(y_test, y_pred_nb, average='weighted'))
print("F1 Score :", f1_score(y_test, y_pred_nb, average='weighted'))

Accuracy : 0.6443678626934253
Precision: 0.6590859087663684
Recall   : 0.6443678626934253
F1 Score : 0.6315743712095482


In [22]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

In [23]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy :", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf, average='weighted'))
print("Recall   :", recall_score(y_test, y_pred_rf, average='weighted'))
print("F1 Score :", f1_score(y_test, y_pred_rf, average='weighted'))

Accuracy : 0.8826272045408473
Precision: 0.8854040267608958
Recall   : 0.8826272045408473
F1 Score : 0.8826021370121806


In [24]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_rf)
    ]
})

print(comparison)

                 Model  Accuracy
0  Logistic Regression  0.680114
1          Naive Bayes  0.644368
2        Random Forest  0.882627


In [25]:
best_model = comparison.loc[comparison["Accuracy"].idxmax()]

print("Best Model:")
print(best_model)

Best Model:
Model       Random Forest
Accuracy         0.882627
Name: 2, dtype: object


In [26]:
new_tweet = ["I really love this game and its graphics"]

# Apply same TF-IDF transformation
new_tweet_tfidf = tfidf.transform(new_tweet)

# Predict
prediction = rf_model.predict(new_tweet_tfidf)

# Convert number back to sentiment label
sentiment = label_encoder.inverse_transform(prediction)

print("Predicted Sentiment:", sentiment[0])

Predicted Sentiment: Positive


In [27]:
new_tweet = ["This game is terrible and full of bugs"]

# Apply same TF-IDF transformation
new_tweet_tfidf = tfidf.transform(new_tweet)

# Predict
prediction = rf_model.predict(new_tweet_tfidf)

# Convert number back to sentiment label
sentiment = label_encoder.inverse_transform(prediction)

print("Predicted Sentiment:", sentiment[0])

Predicted Sentiment: Negative


In [28]:
new_tweet = ["The update was released today"]

# Apply same TF-IDF transformation
new_tweet_tfidf = tfidf.transform(new_tweet)

# Predict
prediction = rf_model.predict(new_tweet_tfidf)

# Convert number back to sentiment label
sentiment = label_encoder.inverse_transform(prediction)

print("Predicted Sentiment:", sentiment[0])

Predicted Sentiment: Irrelevant


In [29]:
val_df = pd.read_csv(
    os.path.join(path, "twitter_validation.csv"),
    header=None
)

val_df.columns = ["Tweet_ID", "Entity", "Sentiment", "Tweet"]

val_df.head()

,Tweet_ID,Entity,Sentiment,Tweet
0,3364,Facebook,Irrelevant,I mentioned on Facebook that I was struggling ...
1,352,Amazon,Neutral,BBC News - Amazon boss Jeff Bezos rejects clai...
2,8312,Microsoft,Negative,@Microsoft Why do I pay for WORD when it funct...
3,4371,CS-GO,Negative,"CSGO matchmaking is so full of closet hacking,..."
4,4433,Google,Neutral,Now the President is slapping Americans in the...


In [30]:
val_df.dropna(inplace=True)

val_df["Tweet"] = val_df["Tweet"].str.lower()

val_df["Tweet"] = val_df["Tweet"].apply(clean_text)

val_df["Tweet"] = val_df["Tweet"].apply(remove_stopwords)

val_df["Tweet"] = val_df["Tweet"].apply(stem_text)

In [31]:
X_val = tfidf.transform(val_df["Tweet"])

y_val = val_df["Sentiment"]

print(X_val.shape)

(1000, 5000)


In [33]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_val_pred = rf_model.predict(X_val)

# Convert y_val to numerical labels using the fitted label_encoder
y_val_encoded = label_encoder.transform(y_val)

print("Validation Accuracy :", accuracy_score(y_val_encoded, y_val_pred))
print("Validation Precision:", precision_score(y_val_encoded, y_val_pred, average='weighted'))
print("Validation Recall   :", recall_score(y_val_encoded, y_val_pred, average='weighted'))
print("Validation F1 Score :", f1_score(y_val_encoded, y_val_pred, average='weighted'))

Validation Accuracy : 0.957
Validation Precision: 0.9574573073246395
Validation Recall   : 0.957
Validation F1 Score : 0.9570276387859551


In [35]:
print("Best Model: Random Forest")

# Convert y_val to numerical labels using the fitted label_encoder
y_val_encoded = label_encoder.transform(y_val)

print("Validation Accuracy :", accuracy_score(y_val_encoded, y_val_pred))
print("Validation Precision:", precision_score(y_val_encoded, y_val_pred, average='weighted'))
print("Validation Recall   :", recall_score(y_val_encoded, y_val_pred, average='weighted'))
print("Validation F1 Score :", f1_score(y_val_encoded, y_val_pred, average='weighted'))

Best Model: Random Forest
Validation Accuracy : 0.957
Validation Precision: 0.9574573073246395
Validation Recall   : 0.957
Validation F1 Score : 0.9570276387859551
